# 02 Data cleaning

**Objective.** Walk through each cleaning rule with before and after counts, using the same clean.py and features.py functions the pipeline runs.

**Inputs.** Raw CSVs via data_prep.clean.load_raw.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from common import style
from common.io import read_parquet, read_table
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
%matplotlib inline

## Step 1. Competition typing and associate sides

In [2]:
from data_prep import clean, features
raw = clean.load_raw()
matches = clean.clean_matches(raw)
matches.groupby(["comp_class", "comp_type"]).size()

comp_class  comp_type     
Emerging    Emerging           30
IPL         IPL               371
T20I        T20I_associate     54
            T20I_full         428
dtype: int64

## Step 2. Exclusion flags and match outcome

In [3]:
matches[["is_no_result", "is_dls", "is_tie", "exclude"]].sum()

is_no_result    40
is_dls          38
is_tie           5
exclude         68
dtype: int64

## Step 3. Ball level rules

In [4]:
balls = clean.clean_balls(raw, matches)
print("rows in", len(raw["ball_by_ball"]), "rows out", len(balls))
balls[["is_legal", "is_wide", "is_noball", "is_bowler_wicket", "is_batter_out", "is_nonstriker_out"]].sum()

rows in 200999 rows out 200999


is_legal             192445
is_wide                7714
is_noball               840
is_bowler_wicket      10272
is_batter_out         10606
is_nonstriker_out       274
dtype: int64

## Step 4. Legal ball validation

In [5]:
reg = balls[balls["innings"] <= 2]
legal = reg[reg["is_legal"]].groupby(["match_id", "innings"]).size()
legal.describe()

count    1712.000000
mean      112.390187
std        16.282453
min         2.000000
25%       113.000000
50%       120.000000
75%       120.000000
max       120.000000
dtype: float64

## Step 5. Dismissal normalisation

In [6]:
pd.concat([raw["ball_by_ball"]["dismissal_type"].value_counts().rename("raw"),
           balls["dismissal_type"].value_counts().rename("clean")], axis=1)

,raw,clean
dismissal_type,,
Caught,7557,7557.0
Bowled,1775,1775.0
LBW,709,709.0
RunOut,600,600.0
Stump,214,214.0
Hit Wicket,17,17.0
Retire Out,6,6.0
Obstracting the field,2,NaN
Obstructing the field,1,3.0


## Step 6. Exclusions table

In [7]:
excl = clean.exclusions_table(matches, balls)
excl

,exclusion,matches,balls
0,No result or abandoned,40,1753
1,"DLS or revised overs, decided",28,4726
2,Super over balls,6,38
3,Kept for ratings and WP,815,194482


## Step 7. Ball state features and row reconciliation

In [8]:
feat = read_parquet("data_prep", "balls_clean.parquet")
print("clean rows", len(feat), "in scope", int(feat["in_scope"].sum()))
feat.loc[feat["in_scope"], ["legal_ball_no", "score_before", "wkts_before", "balls_left", "rrr",
                            "dot_streak_before", "batter_balls_faced_before", "gap_s", "venue_scoring_index"]].describe().T

clean rows 200999 in scope 194482


,count,mean,std,min,25%,50%,75%,max
legal_ball_no,194482.0,58.527925,33.997134,1.000000,29.000000,58.000000,87.000000,120.000000
score_before,194482.0,81.309967,53.295896,0.000000,37.000000,77.000000,120.000000,302.000000
wkts_before,194482.0,2.690768,2.221626,0.000000,1.000000,2.000000,4.000000,9.000000
balls_left,194482.0,62.472075,33.997134,1.000000,34.000000,63.000000,92.000000,120.000000
rrr,93114.0,11.394304,13.983721,0.067416,7.643836,9.612903,12.000000,900.000000
dot_streak_before,194482.0,0.556504,1.015761,0.000000,0.000000,0.000000,1.000000,14.000000
batter_balls_faced_before,194482.0,12.322246,11.397183,0.000000,3.000000,9.000000,18.000000,71.000000
gap_s,162393.0,42.360397,25.200392,5.000000,29.000000,37.000000,46.000000,240.000000
venue_scoring_index,194482.0,1.000358,0.054679,0.879691,0.962722,0.999164,1.045133,1.139388


## Key findings

In [9]:
e = excl.set_index("exclusion")
display(Markdown(f'''
- No rows lost, {len(balls):,} in and out.
- Max legal balls in an innings {legal.max()}, every all-out innings has exactly ten wickets.
- Excluded {e.loc["No result or abandoned", "matches"]} no-result matches and {e.loc["DLS or revised overs, decided", "matches"]} DLS matches, {e.loc["Super over balls", "balls"]} super over balls kept apart.
- {e.loc["Kept for ratings and WP", "matches"]} matches and {e.loc["Kept for ratings and WP", "balls"]:,} balls kept for WP and ratings.
'''))


- No rows lost, 200,999 in and out.
- Max legal balls in an innings 120, every all-out innings has exactly ten wickets.
- Excluded 40 no-result matches and 28 DLS matches, 38 super over balls kept apart.
- 815 matches and 194,482 balls kept for WP and ratings.


## Outputs written

data_prep/output, matches_clean.parquet, balls_clean.parquet, super_overs.parquet and tables/exclusions.csv. Parquet files are gitignored and rebuilt by run_all.